# Vesper Capital — Sequence Model (LSTM) Experiment

Tests whether explicitly modeling temporal sequence structure improves on the XGBoost baseline.

**Hypothesis:** the wallet accumulation signal has temporal dynamics that XGBoost's rolling-window summary features fail to capture. A sequence model (LSTM) sees the full 24-hour history rather than just aggregates and may extract a stronger signal.

**Architecture:**
- Input: 24 hours × ~410 features (raw wallet actions + pool aggregates + price)
- 2-layer LSTM (hidden=64, dropout=0.2)
- 2-layer MLP head with ReLU
- Output: 24h forward return prediction

**Comparison protocol:** same walk-forward folds, same threshold selection, same evaluation — direct apples-to-apples comparison with XGBoost.

**Key design point:** features are intentionally simpler than the XGBoost version. We pass in raw per-hour wallet actions and let the LSTM derive its own temporal patterns, rather than precomputing rolling windows.

**Expected outcomes:**
- If LSTM beats XGBoost by >0.5%: pursue sequence models, refine architecture
- If LSTM comparable: signal is aggregate not sequential — XGBoost is correct
- If LSTM underperforms: confirms aggregate hypothesis; XGBoost wins

**Requirements:** PyTorch. Install with `pip install torch` if needed.

## 1. Configuration

In [ ]:
from claude_refactor_core_utils import *
import pandas as pd
import numpy as np
import os
import calendar
from datetime import date, timedelta
from pathlib import Path
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

# Check for PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    print(f"PyTorch version: {torch.__version__}")
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {DEVICE}")
except ImportError:
    print("PyTorch not installed. Install with:")
    print("  pip install torch")
    raise

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR        = Path('/home/colin/projects/blockchain_alpha')
HOURLY_BAL_CSV  = BASE_DIR / 'data/balance_store_hourly_top200.csv'

# ── Data window ────────────────────────────────────────────────
FULL_START      = date(2023, 1, 1)
FULL_END        = date(2024, 12, 31)
MIN_DELTA_ETH   = 0.005
ROLLING_WINDOWS = [6, 24, 72, 168]

# ── Sequence model config ──────────────────────────────────────
SEQ_LENGTH      = 24       # 24 hours of history per prediction
HIDDEN_DIM      = 64
NUM_LAYERS      = 2
DROPOUT         = 0.2
LEARNING_RATE   = 1e-3
BATCH_SIZE      = 256
N_EPOCHS        = 20
EARLY_STOP_PATIENCE = 5

# ── Walk-forward (matches existing model) ──────────────────────
TRAIN_MONTHS    = 4
VAL_MONTHS      = 2
THRESHOLD_SEARCH = np.arange(0.0, 3.0, 0.05)
MIN_VAL_ACTIVE   = 10

print("=" * 60)
print("VESPER CAPITAL — SEQUENCE MODEL (LSTM)")
print("=" * 60)
print()
print(f"Model: 2-layer LSTM, hidden={HIDDEN_DIM}, dropout={DROPOUT}")
print(f"Sequence length: {SEQ_LENGTH} hours")
print(f"Walk-forward folds: same as XGBoost baseline for direct comparison")


## 2. Load data

Same pipeline as XGBoost — balance store + price data.

In [ ]:
# ============================================================
# Load data — same pipeline as XGBoost walk-forward
# ============================================================

print("Loading balance store...")
raw = pd.read_csv(HOURLY_BAL_CSV)
raw['date']        = pd.to_datetime(raw['date'], format='mixed').dt.floor('h')
raw['address']     = raw['address'].str.lower().str.strip()
raw['balance_wei'] = pd.to_numeric(raw['balance_wei'], errors='coerce').fillna(0).astype('int64')

hourly = (raw.sort_values(['address','date','block_number'])
          .groupby(['address','date'], as_index=False).tail(1)
          .reset_index(drop=True))
hourly['balance_eth'] = hourly['balance_wei'] / 1e18
hourly = hourly.sort_values(['address','date'])
hourly['delta_eth']  = hourly.groupby('address')['balance_eth'].diff()
hourly['action_raw'] = hourly['delta_eth'].apply(
    lambda d: 1 if d > MIN_DELTA_ETH else (-1 if d < -MIN_DELTA_ETH else 0)
)
wallet_list = sorted(hourly['address'].unique().tolist())

print("Loading price data...")
df_price = load_eth_usd_cryptocompare(
    start=str(FULL_START), end=str(FULL_END), interval='1h'
)
df_price['date'] = pd.to_datetime(df_price['date']).dt.floor('h')
df_price = df_price.sort_values('date').reset_index(drop=True)
df_price['fwd_return'] = (
    df_price['close'].shift(-24) - df_price['close']
) / df_price['close'] * 100
df_price['ret_1h']      = df_price['close'].pct_change(1)  * 100
df_price['ret_4h']      = df_price['close'].pct_change(4)  * 100
df_price['ret_24h']     = df_price['close'].pct_change(24) * 100
df_price['vol_24h']     = df_price['ret_1h'].rolling(24).std()

print(f"Wallets: {len(wallet_list)}")
print(f"Hours: {len(df_price):,}")


## 3. Build feature matrix

Simpler than XGBoost version — raw actions, no precomputed rollings. LSTM will derive temporal patterns from the sequence directly.

In [ ]:
# ============================================================
# Build feature matrix — SAME as XGBoost
# Sequence model will internally derive temporal structure
# ============================================================

print("Building feature matrix...")

action_wide = hourly.pivot_table(
    index='date', columns='address', values='action_raw', aggfunc='first'
).fillna(0)
delta_wide = hourly.pivot_table(
    index='date', columns='address', values='delta_eth', aggfunc='first'
).fillna(0)

wf = pd.DataFrame(index=action_wide.index)

# Per-wallet features — just raw action and weighted action
# (Let LSTM derive its own temporal patterns from these)
for addr in wallet_list:
    if addr not in action_wide.columns:
        continue
    short = addr[:10]
    wf[f'{short}_action']     = action_wide[addr]
    wf[f'{short}_wtd_action'] = delta_wide[addr] * action_wide[addr].abs()

# Pool-level features
bm = (action_wide > 0).astype(float)
sm = (action_wide < 0).astype(float)
wf['pool_buy_fraction']  = bm.mean(axis=1)
wf['pool_sell_fraction'] = sm.mean(axis=1)
wf['pool_net_fraction']  = wf['pool_buy_fraction'] - wf['pool_sell_fraction']
bp = delta_wide.clip(lower=0).sum(axis=1)
sp = delta_wide.clip(upper=0).abs().sum(axis=1)
tv = bp + sp + 1e-9
wf['pool_net_pressure_eth'] = (bp - sp) / tv
n_act = (action_wide != 0).sum(axis=1).clip(lower=1)
n_agr = bm.sum(axis=1).combine(sm.sum(axis=1), max)
wf['pool_conviction'] = n_agr / n_act

feature_df = wf.reset_index().rename(columns={'date':'timestamp'})
price_cols = ['date','close','fwd_return','ret_1h','ret_4h','ret_24h','vol_24h']
full_df = feature_df.merge(
    df_price[price_cols].rename(columns={'date':'timestamp'}),
    on='timestamp', how='inner'
).dropna(subset=['fwd_return']).dropna().reset_index(drop=True)

EXCLUDE_COLS = ['timestamp','close','fwd_return']
FEATURE_COLS = [c for c in full_df.columns if c not in EXCLUDE_COLS]

print(f"Features: {len(FEATURE_COLS)}")
print(f"Rows: {len(full_df):,}")
print()
print(f"Key design point: features are SIMPLER than XGBoost version.")
print(f"  No precomputed rolling means (LSTM derives these internally)")
print(f"  Just per-wallet action + weighted action + pool aggregates")
print(f"  LSTM sees {SEQ_LENGTH} hour history → learns the rolling pattern")


## 4. Sequence dataset

Each sample is a (24 hours × n_features) window predicting the 24h forward return.

In [ ]:
# ============================================================
# Sequence dataset
# Each sample: 24-hour feature window → 24h forward return
# ============================================================

class WalletSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_length, target_col='fwd_return'):
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets  = df[target_col].values.astype(np.float32)
        self.timestamps = df['timestamp'].values
        self.seq_length = seq_length

    def __len__(self):
        return max(0, len(self.features) - self.seq_length)

    def __getitem__(self, idx):
        # Return seq_length hours ending at idx+seq_length-1
        # Target is fwd_return at idx+seq_length-1
        x = self.features[idx : idx + self.seq_length]
        y = self.targets[idx + self.seq_length - 1]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)

    def get_timestamp(self, idx):
        return self.timestamps[idx + self.seq_length - 1]


print("Dataset class defined.")
print(f"  Sequence length: {SEQ_LENGTH} hours per sample")
print(f"  Target: 24h forward return at end of sequence")


## 5. LSTM model

In [ ]:
# ============================================================
# LSTM model
# ============================================================

class WalletLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0,
            batch_first = True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        # x: (batch, seq_length, input_dim)
        out, (h_n, c_n) = self.lstm(x)
        # Use final hidden state
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden).squeeze(-1)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_hat = model(x)
        loss  = criterion(y_hat, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    predictions = []
    targets = []
    with torch.no_grad():
        for x, y in loader:
            x, y  = x.to(device), y.to(device)
            y_hat = model(x)
            loss  = criterion(y_hat, y)
            total_loss += loss.item() * len(x)
            predictions.extend(y_hat.cpu().numpy().tolist())
            targets.extend(y.cpu().numpy().tolist())
    return total_loss / len(loader.dataset), np.array(predictions), np.array(targets)


print("Model class defined.")
print(f"  Architecture: 2-layer LSTM (hidden={HIDDEN_DIM}) → 2-layer MLP head")
print(f"  Parameters depend on input_dim (set at fold time)")


## 6. Walk-forward setup

Same fold structure as XGBoost for direct comparison.

In [ ]:
# ============================================================
# Walk-forward with LSTM — same fold structure as XGBoost
# Per fold: train model from scratch, validate, test
# ============================================================

# Build folds (same as XGBoost)
folds = []
cursor = FULL_START
while True:
    train_start = cursor
    train_end_month = train_start.month + TRAIN_MONTHS + VAL_MONTHS - 1
    train_end = date(
        train_start.year + (train_end_month - 1) // 12,
        (train_end_month - 1) % 12 + 1,
        calendar.monthrange(
            train_start.year + (train_end_month - 1) // 12,
            (train_end_month - 1) % 12 + 1
        )[1]
    )
    test_start = train_end + timedelta(days=1)
    test_end   = date(
        test_start.year + (test_start.month - 1) // 12,
        test_start.month,
        calendar.monthrange(test_start.year, test_start.month)[1]
    )
    if test_end > FULL_END:
        break
    folds.append({
        'train_start': train_start, 'train_end': train_end,
        'test_start':  test_start,  'test_end':  test_end,
    })
    next_month = cursor.month + 1
    next_year  = cursor.year + (next_month - 1) // 12
    next_month = (next_month - 1) % 12 + 1
    cursor = date(next_year, next_month, 1)

print(f"Walk-forward folds: {len(folds)}")

# Standardize features (fit on training, apply to val/test)
from sklearn.preprocessing import StandardScaler

def standardize_fold(df_train, df_val, df_test, feature_cols):
    scaler = StandardScaler()
    train_features = scaler.fit_transform(df_train[feature_cols].values)
    val_features   = scaler.transform(df_val[feature_cols].values)
    test_features  = scaler.transform(df_test[feature_cols].values)

    df_train_scaled = df_train.copy()
    df_val_scaled   = df_val.copy()
    df_test_scaled  = df_test.copy()
    df_train_scaled[feature_cols] = train_features
    df_val_scaled[feature_cols]   = val_features
    df_test_scaled[feature_cols]  = test_features
    return df_train_scaled, df_val_scaled, df_test_scaled

print("Helper functions ready.")


## 7. Run walk-forward

This will take longer than XGBoost — expect 10-30 minutes depending on hardware. GPU strongly recommended; CPU works but is slow.

In [ ]:
print("Starting LSTM walk-forward...")
print()
torch.manual_seed(42)
np.random.seed(42)

results = []
input_dim = len(FEATURE_COLS)

for fold_idx, fold in enumerate(folds):
    val_start        = fold['train_end'] - relativedelta(months=2) + relativedelta(days=1)
    train_end_actual = val_start - relativedelta(days=1)

    train_mask = ((full_df['timestamp'].dt.date >= fold['train_start']) &
                  (full_df['timestamp'].dt.date <= train_end_actual))
    val_mask   = ((full_df['timestamp'].dt.date >= val_start) &
                  (full_df['timestamp'].dt.date <= fold['train_end']))
    test_mask  = ((full_df['timestamp'].dt.date >= fold['test_start']) &
                  (full_df['timestamp'].dt.date <= fold['test_end']))

    train_fold = full_df[train_mask].copy().sort_values('timestamp').reset_index(drop=True)
    val_fold   = full_df[val_mask].copy().sort_values('timestamp').reset_index(drop=True)
    test_fold  = full_df[test_mask].copy().sort_values('timestamp').reset_index(drop=True)

    if len(train_fold) < SEQ_LENGTH + 200 or len(val_fold) < SEQ_LENGTH + 100 or len(test_fold) < SEQ_LENGTH + 50:
        print(f"Fold {fold_idx+1}: insufficient data, skipping")
        continue

    # Standardize
    train_s, val_s, test_s = standardize_fold(train_fold, val_fold, test_fold, FEATURE_COLS)

    # Datasets
    train_ds = WalletSequenceDataset(train_s, FEATURE_COLS, SEQ_LENGTH)
    val_ds   = WalletSequenceDataset(val_s,   FEATURE_COLS, SEQ_LENGTH)
    test_ds  = WalletSequenceDataset(test_s,  FEATURE_COLS, SEQ_LENGTH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model     = WalletLSTM(input_dim, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

    # Train with early stopping
    best_val_loss   = float('inf')
    patience_count  = 0
    best_state      = None
    for epoch in range(N_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            patience_count  = 0
            best_state      = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= EARLY_STOP_PATIENCE:
                break

    # Restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # Threshold selection on validation
    _, val_preds, val_targets = evaluate(model, val_loader, criterion, DEVICE)
    base_val_mean = val_targets.mean()

    best_threshold = 0.0
    best_val_edge  = -999
    for threshold in THRESHOLD_SEARCH:
        pos = (val_preds > threshold).astype(float)
        n_act = int(pos.sum())
        if n_act < MIN_VAL_ACTIVE:
            continue
        active_returns = val_targets[pos > 0]
        edge = active_returns.mean() - base_val_mean
        if edge > best_val_edge:
            best_val_edge  = edge
            best_threshold = float(threshold)

    # Apply to test
    _, test_preds, test_targets = evaluate(model, test_loader, criterion, DEVICE)
    test_pos      = (test_preds > best_threshold).astype(float)
    base_test_mean = test_targets.mean()
    active_test   = test_targets[test_pos > 0]

    if len(active_test) > 0:
        edge_test = active_test.mean() - base_test_mean
        win_rate  = (active_test > 0).mean()
    else:
        edge_test = 0.0
        win_rate  = 0.0

    print(f"Fold {fold_idx+1:2d}: {fold['test_start']}  "
          f"thr={best_threshold:.3f}  "
          f"n={len(active_test):4d}  "
          f"edge={edge_test:+.4f}%  "
          f"win={win_rate:.3f}  "
          f"epochs={epoch+1}")

    results.append({
        'fold':       fold_idx + 1,
        'test_month': fold['test_start'].strftime('%Y-%m'),
        'test_start': fold['test_start'],
        'threshold':  best_threshold,
        'n_active':   len(active_test),
        'edge':       edge_test,
        'win_rate':   win_rate,
        'is_november': fold['test_start'].month == 11,
        'epochs_trained': epoch + 1,
    })

results_df = pd.DataFrame(results)
print(f"\nWalk-forward complete: {len(results_df)} folds")


## 8. Compare LSTM vs XGBoost

In [ ]:
# ============================================================
# Compare LSTM walk-forward vs XGBoost baseline
# ============================================================

from scipy import stats as scipy_stats

print("=" * 65)
print("LSTM WALK-FORWARD RESULTS")
print("=" * 65)
print()
print(results_df[['test_month','threshold','n_active','edge',
                  'win_rate','is_november','epochs_trained']
                ].to_string(index=False))
print()

active   = results_df[results_df['n_active'] >= 5]
non_nov  = active[~active['is_november']]

print(f"Total folds:              {len(results_df)}")
print(f"Active folds (n>=5):      {len(active)}")
print(f"Non-November active:      {len(non_nov)}")
print()

if len(non_nov) > 0:
    edges  = non_nov['edge'].values
    weights = non_nov['n_active'].values
    print(f"Non-November (LSTM):")
    print(f"  Unweighted mean edge: {np.mean(edges):+.4f}%/hr")
    print(f"  Weighted mean edge:   {np.average(edges, weights=weights):+.4f}%/hr")
    print(f"  Median edge:          {np.median(edges):+.4f}%/hr")
    print(f"  Positive folds:       {(edges > 0).sum()} / {len(edges)}")
    print()

    if len(non_nov) >= 3:
        t, p = scipy_stats.ttest_1samp(edges, 0)
        sig = '✓ SIGNIFICANT' if p < 0.05 else '✗ not significant'
        print(f"T-test non-November:    t={t:.3f}  p={p:.4f}  {sig}")

print()
print("=" * 65)
print("COMPARISON vs XGBoost BASELINE")
print("=" * 65)
print()
print(f"{'Metric':<35} {'XGBoost':>12} {'LSTM':>12}")
print("-" * 60)
print(f"{'Non-Nov mean edge (unweighted)':<35} {'+1.0045%':>12} "
      f"{f'{np.mean(edges):+.4f}%' if len(non_nov)>0 else 'N/A':>12}")
print(f"{'Non-Nov mean edge (weighted)':<35} {'+0.2156%':>12} "
      f"{f'{np.average(edges, weights=weights):+.4f}%' if len(non_nov)>0 else 'N/A':>12}")
print(f"{'Non-Nov positive folds':<35} {'9/12':>12} "
      f"{f'{(edges>0).sum()}/{len(edges)}' if len(non_nov)>0 else 'N/A':>12}")
print(f"{'Non-Nov t-test p-value':<35} {'0.0286':>12} "
      f"{f'{p:.4f}' if len(non_nov)>=3 else 'N/A':>12}")
print()

if len(non_nov) >= 3:
    improvement = np.mean(edges) - 1.0045
    if improvement > 0.5:
        verdict = "✓ LSTM materially outperforms XGBoost — sequence model captures the signal better"
    elif improvement > 0:
        verdict = "~ LSTM modestly outperforms XGBoost — worth pursuing further"
    elif improvement > -0.5:
        verdict = "~ LSTM comparable to XGBoost — sequence structure adds little"
    else:
        verdict = "✗ LSTM underperforms XGBoost — XGBoost remains the better choice"
    print(verdict)
    print()
    print("Interpretation:")
    if improvement > 0:
        print("  The LSTM is capturing temporal patterns that XGBoost's rolling-window")
        print("  features miss. The signal has genuine sequence structure beyond what")
        print("  summary statistics capture.")
    else:
        print("  Despite explicit sequence modeling, the LSTM does not outperform")
        print("  XGBoost's engineered features. This suggests the wallet signal is")
        print("  fundamentally aggregate (is anyone buying?) rather than sequential")
        print("  (what does the buying pattern look like?). Stick with XGBoost.")

results_df.to_csv(BASE_DIR / 'data/walkforward_lstm_results.csv', index=False)
print()
print(f"Results saved to: walkforward_lstm_results.csv")
